# HTML Video

Browse bundled HTML video templates, inspect a template, and render configured WebM frames or media ports through the app MCP tools.

In [6]:
const encoder = new TextEncoder();
const decoder = new TextDecoder();

async function readExactly(reader: { read(p: Uint8Array): Promise<number | null> }, size: number): Promise<Uint8Array> {
  const buffer = new Uint8Array(size);
  let offset = 0;
  while (offset < size) {
    const n = await reader.read(buffer.subarray(offset));
    if (n === null) throw new Error("notebook MCP socket closed");
    offset += n;
  }
  return buffer;
}

async function readFrame(conn: Deno.Conn): Promise<any> {
  const header = await readExactly(conn, 4);
  const length = new DataView(header.buffer, header.byteOffset, header.byteLength).getUint32(0, false);
  return JSON.parse(decoder.decode(await readExactly(conn, length)));
}

async function writeFrame(conn: Deno.Conn, value: unknown): Promise<void> {
  const payload = encoder.encode(JSON.stringify(value));
  const frame = new Uint8Array(4 + payload.length);
  new DataView(frame.buffer).setUint32(0, payload.length, false);
  frame.set(payload, 4);
  await conn.write(frame);
}

async function callNotebookTool(name: string, args: Record<string, unknown>): Promise<any> {
  const socketPath = Deno.env.get("SPUR_NOTEBOOK_MCP_SOCKET");
  if (!socketPath) throw new Error("SPUR_NOTEBOOK_MCP_SOCKET is not set");
  const conn = await Deno.connect({ transport: "unix", path: socketPath });
  let id = 1;
  try {
    await writeFrame(conn, {
      jsonrpc: "2.0",
      id: id++,
      method: "initialize",
      params: {
        protocolVersion: "2025-11-25",
        capabilities: {},
        clientInfo: { name: "html-video-app", version: "0" }
      }
    });
    await readFrame(conn);
    await writeFrame(conn, { jsonrpc: "2.0", method: "notifications/initialized", params: {} });
    const requestId = id++;
    await writeFrame(conn, { jsonrpc: "2.0", id: requestId, method: "tools/call", params: { name, arguments: args } });
    const response = await readFrame(conn);
    if (response.error) throw new Error(response.error.message ?? JSON.stringify(response.error));
    const result = response.result ?? {};
    if (result.structuredContent) return result.structuredContent;
    if (result.structured_content) return result.structured_content;
    const text = result.content?.find?.((item: any) => item.type === "text")?.text;
    if (typeof text === "string") {
      try { return JSON.parse(text); } catch (_) { return { text }; }
    }
    return result;
  } finally {
    try { conn.close(); } catch (_) {}
  }
}

const searchResult = await callNotebookTool("html_video_search_templates", {
  intent: "product launch motion typography dashboard intro",
  top: 8
});
const items = Array.isArray(searchResult.items) ? searchResult.items : [];
const selectedId = items[0]?.id ?? "basic";
const { tableFromArrays } = await import("npm:apache-arrow@21.1.0");
const searchTable = tableFromArrays({
  id: items.map((i: any) => i.id ?? ""),
  title: items.map((i: any) => i.title ?? ""),
  intent: items.map((i: any) => i.intent ?? ""),
  summary: items.map((i: any) => i.summary ?? ""),
  score: items.map((i: any) => Number(i.score ?? 0)),
});
await spur.put("template_search", searchTable);

function readPortPayload(port: string): number[] | null {
  const root = Deno.env.get("SPUR_NOTEBOOK_PORT_ROOT");
  if (!root) return null;
  const manifest = JSON.parse(Deno.readTextFileSync(`${root}/ports/manifest.json`));
  const path = manifest.ports?.[port]?.path;
  if (!path) return null;
  return Array.from(Deno.readFileSync(path));
}

const selectionPayloads: Record<string, number[]> = {};
for (const item of items) {
  if (!item?.id) continue;
  await spur.put("template_selection", [{ id: item.id }]);
  const payload = readPortPayload("template_selection");
  if (payload) selectionPayloads[item.id] = payload;
}
await spur.put("template_selection", [{ id: selectedId }]);

const { widget } = await spur.anywidget();
widget({
  state: { items, selectedId, selectionPayloads },
  render({ model, el, experimental }: any) {
    const items = model.get("items") ?? [];
    const selectedId = model.get("selectedId");
    el.innerHTML = `
      <style>
        .hv-search{padding:24px;font:14px system-ui,sans-serif;color:#172033;background:#fbfcfe}
        .hv-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px;margin-top:16px}
        .hv-card{border:1px solid #d9e2ec;background:white;border-radius:8px;padding:14px;box-shadow:0 1px 2px rgba(15,23,42,.04);cursor:pointer;text-align:left}
        .hv-card[data-selected="true"]{border-color:#2563eb;box-shadow:0 0 0 2px rgba(37,99,235,.16)}
        .hv-title{font-weight:700;margin-bottom:6px}.hv-tags{display:flex;gap:6px;flex-wrap:wrap;margin-top:10px}.hv-tag{font-size:11px;border:1px solid #dbe3ef;border-radius:999px;padding:2px 7px;background:#f8fafc}
      </style>
      <section class="hv-search">
        <h2>Template Browser</h2>
        <div class="hv-grid">
          ${items.map((item: any) => `<article class="hv-card" data-template-id="${item.id}" data-selected="${item.id === selectedId}"><div class="hv-title">${item.title ?? item.id}</div><div>${item.summary ?? item.intent ?? ""}</div><div class="hv-tags">${(item.tags ?? []).map((tag: any) => `<span class="hv-tag">${tag}</span>`).join("")}</div></article>`).join("")}
        </div>
      </section>`;
    el.querySelectorAll(".hv-card").forEach((card: any) => {
      card.addEventListener("click", async () => {
        const id = card.getAttribute("data-template-id");
        if (!id) return;
        model.set("selectedId", id);
        el.querySelectorAll(".hv-card").forEach((candidate: any) => {
          candidate.dataset.selected = String(candidate.getAttribute("data-template-id") === id);
        });
        const payload = model.get("selectionPayloads")?.[id];
        if (payload && experimental?.invoke) {
          try { await experimental.invoke("source.push", { port: "template_selection", payload }); } catch (_) {}
        }
      });
    });
  }
})


id,title,intent,summary,score
frame-liquid-hero,Liquid Hero,Create a polished hero or opener frame with a flowing animated gradient background and centered headline.,"Self-contained canvas gradient hero frame with headline, subtext, and soft liquid motion.",1.5


id
frame-liquid-hero


id
frame-liquid-hero


In [7]:
const encoder = new TextEncoder();
const decoder = new TextDecoder();

async function readExactly(reader: { read(p: Uint8Array): Promise<number | null> }, size: number): Promise<Uint8Array> {
  const buffer = new Uint8Array(size);
  let offset = 0;
  while (offset < size) {
    const n = await reader.read(buffer.subarray(offset));
    if (n === null) throw new Error("notebook MCP socket closed");
    offset += n;
  }
  return buffer;
}

async function readFrame(conn: Deno.Conn): Promise<any> {
  const header = await readExactly(conn, 4);
  const length = new DataView(header.buffer, header.byteOffset, header.byteLength).getUint32(0, false);
  return JSON.parse(decoder.decode(await readExactly(conn, length)));
}

async function writeFrame(conn: Deno.Conn, value: unknown): Promise<void> {
  const payload = encoder.encode(JSON.stringify(value));
  const frame = new Uint8Array(4 + payload.length);
  new DataView(frame.buffer).setUint32(0, payload.length, false);
  frame.set(payload, 4);
  await conn.write(frame);
}

async function callNotebookTool(name: string, args: Record<string, unknown>): Promise<any> {
  const socketPath = Deno.env.get("SPUR_NOTEBOOK_MCP_SOCKET");
  if (!socketPath) throw new Error("SPUR_NOTEBOOK_MCP_SOCKET is not set");
  const conn = await Deno.connect({ transport: "unix", path: socketPath });
  let id = 1;
  try {
    await writeFrame(conn, {
      jsonrpc: "2.0",
      id: id++,
      method: "initialize",
      params: {
        protocolVersion: "2025-11-25",
        capabilities: {},
        clientInfo: { name: "html-video-app", version: "0" }
      }
    });
    await readFrame(conn);
    await writeFrame(conn, { jsonrpc: "2.0", method: "notifications/initialized", params: {} });
    const requestId = id++;
    await writeFrame(conn, { jsonrpc: "2.0", id: requestId, method: "tools/call", params: { name, arguments: args } });
    const response = await readFrame(conn);
    if (response.error) throw new Error(response.error.message ?? JSON.stringify(response.error));
    const result = response.result ?? {};
    if (result.structuredContent) return result.structuredContent;
    if (result.structured_content) return result.structured_content;
    const text = result.content?.find?.((item: any) => item.type === "text")?.text;
    if (typeof text === "string") {
      try { return JSON.parse(text); } catch (_) { return { text }; }
    }
    return result;
  } finally {
    try { conn.close(); } catch (_) {}
  }
}

function normalizeArrowValue(value: any): any {
  if (value === null || value === undefined) return value;
  if (typeof value.toJSON === "function") return value.toJSON();
  if (Array.isArray(value)) return value.map(normalizeArrowValue);
  if (typeof value === "object") {
    const output: Record<string, unknown> = {};
    for (const [key, item] of Object.entries(value)) output[key] = normalizeArrowValue(item);
    return output;
  }
  return value;
}

function tableRows(table: any): any[] {
  if (!table) return [];
  return table.toArray().map((row: any) => normalizeArrowValue(row));
}

function firstPortRow(port: string): any | null {
  try {
    return tableRows(spur.get(port))[0] ?? null;
  } catch (_) {
    return null;
  }
}

const search = firstPortRow("template_search") ?? { items: [] };
const items = Array.isArray(search.items) ? search.items : [];
const selection = firstPortRow("template_selection");
const templateId = selection?.id ?? items[0]?.id ?? "basic";
const template = await callNotebookTool("html_video_get_template", { id: templateId });
const { tableFromArrays: _tfa2 } = await import("npm:apache-arrow@21.1.0");
const templateTable = _tfa2({
  id: [template.metadata?.id ?? template.id ?? templateId],
  title: [template.metadata?.title ?? template.id ?? templateId],
  summary: [template.metadata?.summary ?? ""],
  html: [template.html ?? ""],
  skill_md: [template.skill_md ?? ""],
});
await spur.put("template_data", templateTable);

const { widget } = await spur.anywidget();
widget({
  state: {
    title: template.metadata?.title ?? template.id ?? templateId,
    id: template.metadata?.id ?? template.id ?? templateId,
    summary: template.metadata?.summary ?? "",
    html: template.html ?? "",
    skill: template.skill_md ?? ""
  },
  render({ model, el }: any) {
    const html = model.get("html") ?? "";
    el.innerHTML = `
      <style>
        .hv-preview{display:grid;grid-template-columns:minmax(280px,1fr) minmax(280px,1.2fr);gap:18px;padding:24px;font:14px system-ui,sans-serif;color:#172033;background:white}
        .hv-pane{min-height:220px;border:1px solid #d9e2ec;border-radius:8px;background:#fbfcfe;overflow:hidden}.hv-meta{padding:18px}.hv-meta h2{margin:0 0 8px}.hv-code{margin-top:14px;max-height:180px;overflow:auto;border:1px solid #e2e8f0;background:#0f172a;color:#e2e8f0;border-radius:6px;padding:12px;font:12px ui-monospace,monospace;white-space:pre-wrap}.hv-frame{width:100%;height:100%;min-height:360px;border:0;background:white}
      </style>
      <section class="hv-preview">
        <div class="hv-pane hv-meta"><h2>${model.get("title")}</h2><div>${model.get("summary")}</div><pre class="hv-code">${html.replace(/[<&]/g, (c: string) => c === "<" ? "&lt;" : "&amp;")}</pre></div>
        <div class="hv-pane"><iframe class="hv-frame" sandbox="allow-scripts" srcdoc="${html.replace(/&/g, "&amp;").replace(/"/g, "&quot;")}"></iframe></div>
      </section>`;
  }
})


SPUR port template_data v5 1 rows x 5 columns id title summary html skill_md frame-liquid-hero Liquid Hero Self-contained canvas gradient hero frame with headline, subtext, and soft liquid motion. <!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Liquid Hero</title>
<style>
html,body{margin:0;height:100%;overflow:hidden;background:#071019}
canvas{display:block;width:100vw;height:100vh}
</style>
</head>
<body>
<canvas data-capture="true" aria-label="Animated liquid gradient hero"></canvas>
<script>
const canvas=document.querySelector("canvas");
const ctx=canvas.getContext("2d");
const copy={
 eyebrow:"Now entering",
 headline:"Fluid Strategy",
 subtext:"A calm, cinematic opener for product narratives, creative briefs, and section transitions."
};
function resize(){
 canvas.width = window.innerWidth;
 canvas.height = window.innerHeight;
}
window.addEventListener("resize",resize);
resize();
function fit(size,text,maxWidth){
 ctx.font=`900 ${size}px Arial, Helvetica, sans-serif`;
 while(size>34&&ctx.measureText(text).width>maxWidth){
 size-=4; ctx.font=`900 ${size}px Arial, Helvetica, sans-serif`;
 }
 return size;
}
function wrap(text,x,y,maxWidth,size,lineHeight){
 ctx.font=`700 ${size}px Arial, Helvetica, sans-serif`;
 const words=text.split(" "); let line="";
 for(const word of words){
 const test=line?`${line} ${word}`:word;
 if(ctx.measureText(test).width>maxWidth&&line){ctx.fillText(line,x,y); y+=lineHeight; line=word;}
 else line=test;
 }
 if(line)ctx.fillText(line,x,y);
}
function blob(x,y,r,color){
 const g=ctx.createRadialGradient(x,y,0,x,y,r);
 g.addColorStop(0,color); g.addColorStop(.58,color.replace(/[\d.]+\)$/,".18)")); g.addColorStop(1,"transparent");
 ctx.fillStyle=g; ctx.fillRect(0,0,canvas.width,canvas.height);
}
function frame(now){
 const t=now/1000,w=canvas.width,h=canvas.height,cx=w/2,cy=h/2;
 const bg=ctx.createLinearGradient(0,0,w,h);
 bg.addColorStop(0,"#08111f"); bg.addColorStop(.28,"#0f766e"); bg.addColorStop(.55,"#155e75");
 bg.addColorStop(.78,"#7c3aed"); bg.addColorStop(1,"#111827"); ctx.fillStyle=bg; ctx.fillRect(0,0,w,h);
 ctx.globalCompositeOperation="screen";
 blob(cx+Math.sin(t*.62)*w*.18,cy+Math.cos(t*.48)*h*.16,Math.max(w,h)*.55,"rgba(103,232,249,.66)");
 blob(cx+Math.cos(t*.50)*w*.22,cy+Math.sin(t*.55)*h*.18,Math.max(w,h)*.50,"rgba(240,171,252,.58)");
 blob(cx+Math.sin(t*.34+2)*w*.25,cy+Math.cos(t*.41+1)*h*.12,Math.max(w,h)*.42,"rgba(167,243,208,.42)");
 ctx.globalCompositeOperation="source-over";
 const wash=ctx.createLinearGradient(0,0,0,h); wash.addColorStop(0,"rgba(0,0,0,.08)"); wash.addColorStop(1,"rgba(0,0,0,.28)");
 ctx.fillStyle=wash; ctx.fillRect(0,0,w,h);
 ctx.textAlign="center"; ctx.textBaseline="middle";
 ctx.shadowColor="rgba(0,0,0,.28)"; ctx.shadowBlur=24;
 ctx.font=`800 ${Math.max(15,Math.min(26,w*.018))}px Arial, Helvetica, sans-serif`;
 ctx.fillStyle="#bae6fd"; ctx.fillText(copy.eyebrow.toUpperCase(),cx,cy-h*.18+Math.sin(t*1.2)*3);
 const titleSize=fit(Math.max(58,Math.min(150,w*.10)),copy.headline,w*.88);
 ctx.font=`900 ${titleSize}px Arial, Helvetica, sans-serif`; ctx.fillStyle="#ffffff";
 ctx.fillText(copy.headline,cx,cy);
 ctx.fillStyle="#e0f2fe"; wrap(copy.subtext,cx,cy+h*.18,Math.min(760,w*.82),Math.max(18,Math.min(31,w*.022)),Math.max(27,w*.032));
 ctx.shadowBlur=0;
 const pulse=.72+Math.sin(t*2.2)*.18,barW=w*.58*pulse;
 const line=ctx.createLinearGradient(cx-barW/2,0,cx+barW/2,0);
 line.addColorStop(0,"transparent"); line.addColorStop(.35,"#a7f3d0"); line.addColorStop(.65,"#f0abfc"); line.addColorStop(1,"transparent");
 ctx.fillStyle=line; ctx.fillRect(cx-barW/2,h*.88,barW,2);
 requestAnimationFrame(frame);
}
requestAnimationFrame(frame);
</script>
</body>
</html>
 ---
name: frame-liquid-hero
description: Canvas animated liquid-gradient hero frame for polished openers, transitions, and calm headline moments.
---
# Frame: Liquid Hero

Use this template when a video needs a refined h

In [15]:
const encoder = new TextEncoder();
const decoder = new TextDecoder();

async function readExactly(reader: { read(p: Uint8Array): Promise<number | null> }, size: number): Promise<Uint8Array> {
  const buffer = new Uint8Array(size);
  let offset = 0;
  while (offset < size) {
    const n = await reader.read(buffer.subarray(offset));
    if (n === null) throw new Error("notebook MCP socket closed");
    offset += n;
  }
  return buffer;
}

async function readFrame(conn: Deno.Conn): Promise<any> {
  const header = await readExactly(conn, 4);
  const length = new DataView(header.buffer, header.byteOffset, header.byteLength).getUint32(0, false);
  return JSON.parse(decoder.decode(await readExactly(conn, length)));
}

async function writeFrame(conn: Deno.Conn, value: unknown): Promise<void> {
  const payload = encoder.encode(JSON.stringify(value));
  const frame = new Uint8Array(4 + payload.length);
  new DataView(frame.buffer).setUint32(0, payload.length, false);
  frame.set(payload, 4);
  await conn.write(frame);
}

async function callNotebookTool(name: string, args: Record<string, unknown>): Promise<any> {
  const socketPath = Deno.env.get("SPUR_NOTEBOOK_MCP_SOCKET");
  if (!socketPath) throw new Error("SPUR_NOTEBOOK_MCP_SOCKET is not set");
  const conn = await Deno.connect({ transport: "unix", path: socketPath });
  let id = 1;
  try {
    await writeFrame(conn, {
      jsonrpc: "2.0",
      id: id++,
      method: "initialize",
      params: {
        protocolVersion: "2025-11-25",
        capabilities: {},
        clientInfo: { name: "html-video-app", version: "0" }
      }
    });
    await readFrame(conn);
    await writeFrame(conn, { jsonrpc: "2.0", method: "notifications/initialized", params: {} });
    const requestId = id++;
    await writeFrame(conn, { jsonrpc: "2.0", id: requestId, method: "tools/call", params: { name, arguments: args } });
    const response = await readFrame(conn);
    if (response.error) throw new Error(response.error.message ?? JSON.stringify(response.error));
    const result = response.result ?? {};
    if (result.structuredContent) return result.structuredContent;
    if (result.structured_content) return result.structured_content;
    const text = result.content?.find?.((item: any) => item.type === "text")?.text;
    if (typeof text === "string") {
      try { return JSON.parse(text); } catch (_) { return { text }; }
    }
    return result;
  } finally {
    try { conn.close(); } catch (_) {}
  }
}

function normalizeArrowValue(value: any): any {
  if (value === null || value === undefined) return value;
  if (typeof value.toJSON === "function") return value.toJSON();
  if (Array.isArray(value)) return value.map(normalizeArrowValue);
  if (typeof value === "object") {
    const output: Record<string, unknown> = {};
    for (const [key, item] of Object.entries(value)) output[key] = normalizeArrowValue(item);
    return output;
  }
  return value;
}

function firstPortRow(port: string): any | null {
  try {
    return spur.get(port).toArray().map((row: any) => normalizeArrowValue(row))[0] ?? null;
  } catch (_) {
    return null;
  }
}

const template = firstPortRow("template_data");
const templateId = template?.metadata?.id ?? template?.id ?? "basic";
const defaultOutput = `${Deno.cwd()}/html-video-render.mp4`;
const request = null;
let renderResult = null;
let renderError = null;
if (request) {
  try {
    renderResult = await callNotebookTool("html_video_render", request);
  } catch (error) {
    renderError = error instanceof Error ? error.message : String(error);
  }
}

const { widget } = await spur.anywidget();
widget({
  state: { request, renderResult, renderError, defaultOutput, templateId },
  render({ model, el }: any) {
    const request = model.get("request");
    const result = model.get("renderResult");
    const error = model.get("renderError");
    el.innerHTML = `
      <style>
        .hv-render{padding:24px;font:14px system-ui,sans-serif;color:#172033;background:#f8fafc}.hv-panel{max-width:760px;border:1px solid #d9e2ec;border-radius:8px;background:white;padding:18px}.hv-row{display:grid;grid-template-columns:150px 1fr;gap:10px;align-items:center;margin-top:10px}.hv-input{border:1px solid #cbd5e1;border-radius:6px;padding:8px;background:#f8fafc;font:13px ui-monospace,monospace}.hv-status{margin-top:16px;border-radius:6px;padding:12px;background:#eef6ff;border:1px solid #bfdbfe}.hv-error{background:#fff1f2;border-color:#fecdd3}.hv-json{white-space:pre-wrap;font:12px ui-monospace,monospace}</style>
      <section class="hv-render"><div class="hv-panel"><h2>Render Controls</h2><div class="hv-row"><label>Template</label><div class="hv-input">${model.get("templateId")}</div></div><div class="hv-row"><label>Output path</label><div class="hv-input">${request?.output_path ?? model.get("defaultOutput")}</div></div><div class="hv-row"><label>Resolution</label><div class="hv-input">${request?.resolution ?? "1280x720"}</div></div><div class="hv-row"><label>FPS</label><div class="hv-input">${request?.fps ?? 30}</div></div>${result ? `<div class="hv-status"><strong>Rendered</strong><pre class="hv-json">${JSON.stringify(result, null, 2)}</pre></div>` : error ? `<div class="hv-status hv-error"><strong>Render error</strong><pre class="hv-json">${error}</pre></div>` : `<div class="hv-status"><strong>Ready</strong><div>Render request is waiting for webm_frames or port_names.</div></div>`}</div></section>`;
  }
})
